# Surface-Based Source Spaces

The Source Space is a distributed dipole model. Brain sources are approximated by dipoles and placed uniformly along the brain surface (more specifically, the white surface — the boundary between gray and white matter). The number of dipoles is adjustable based on the `spacing`. We then eliminate any dipole that is not located within the gray matter, as only the synaptic currents occurring in the gray matter are believed to contribute to EEG.

<img src="../../../img/sourcespace.png">

## 📦 Imports

In [1]:
import mindmap as mm
import mne

# readers
dm = mm.io.TutoManager("../../config.yml")
sr = dm.get_subject("intra-12")

## 🧠 Data

📥 **<font color=pink>Inputs</font>**

**Anatomy**
- FREESURFER reconstruction folder.
    - `white`: white surface
- `ctx`: Cortical Mask.

**Transformations**
- `tkras2ras`: Affine transform (freesurfer ras -> T1 ras)

🎚️ **<font color=pink>Configuration</font>**:
- `fconfig`: Configuration file.
- `sub`: Subject id (position in list SUBS)

In [5]:
# filter source space with nifti mask if exists
filtering = "nifti"

try:
    ctx = sr.load("ctx")
    trans = sr.load("tkras2ras")
except FileNotFoundError:
    vertex_ctx = sr.load("vertex-ctx")
    filtering = "vertex-id"

## ⚙️ Build Source Spaces

MNE selects dipoles on the white matter surface by uniformly downsampling the white mesh. The parameter “octX” defines the downsampling level of the triangular mesh.


📤 **<font color=pink>Outputs</font>**
- `source_space`: MNE object containing dipole locations and dipole distances to each other.

🎚️ **<font color=pink>Setup source space</font>**:
- `spacing`: Spacing in [mm] between dipoles or mesh downsampling level.
- `n_jobs`: Number of parrallel jobs

In [8]:
# dipole spacing ("octN is a mesh downsampling method")
spacing = "oct6"

# number of threads
n_jobs = 10


# Setup source space
source_space = mne.setup_source_space(
    subject      = f"sub-{sr.sub}",
    spacing      = spacing,
    surface      = "white",
    add_dist     = True,                    # geodesic distances between sources
    subjects_dir = dm.config["DIR_FS"],
    n_jobs       = n_jobs,
    verbose      = "warning",
)
    
# gray matter constraint
if filtering == "nifti":
    source_space = mm.anatomy.source_space.filter_dipoles_by_nifti_mask(source_space, ctx, trans)
else:
    source_space = mm.anatomy.source_space.filter_dipoles_by_id_list(source_space, vertex_ctx)

print("\033[92mSUMMARY\033[0m")
for src in source_space:
    print(f"Source space id: {src['id']}")
    print(f"Type: {src['type']}")
    print(f"Number of sources: {src['nuse']}\n")

SUMMARY
Source space id: 101 (FIFFV_MNE_SURF_LEFT_HEMI)
Type: surf
Number of sources: 3821

Source space id: 102 (FIFFV_MNE_SURF_RIGHT_HEMI)
Type: surf
Number of sources: 3826



📊 **Source space** <br>

In [9]:
if filtering == "nifti":
    mm.viz.plot_brain(source_space, show_src=True, overlay=ctx, tkras2ras=trans, overlay_color="gray", overlay_opacity=0.5).show()
else:
    mm.viz.plot_brain(source_space, mesh_params={"show_src":True}).show()

Widget(value='<iframe src="http://localhost:38583/index.html?ui=P_0x70e4ec526190_0&reconnect=auto" class="pyvi…